# 1. Initializations

## 1.1 General CPU/GPU Checks (NVIDIA cards)

In [ ]:
### global
import logging
import os
import shutil
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
import os
print(f'Path [{os.environ["PATH"]}]')

# torch_test.py
import torch
print(f"✅ Torch CUDA available: {torch.cuda.is_available()}")
print(f"🖥️ Device: {torch.cuda.get_device_name(0)}")

# tf_test.py
import tensorflow as tf
print("✅ TF GPU:", tf.config.list_physical_devices("GPU"))

## 1.2 General imports

In [ ]:
# Pour la manipulation de tableaux et Dataframes
import numpy as np
import itertools 

# Pour les modèles et leur preprocessing
from sklearn.metrics import classification_report, confusion_matrix

# Pour la visualisation des performances
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Pour construire un réseau de neurone
from tensorflow.keras import Sequential, callbacks
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import Input, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import mnist
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from timeit import default_timer as timer

# Pour la transformation sur les images
from tensorflow.keras.layers import Rescaling
from tensorflow.keras.layers import Resizing
from tensorflow.keras.layers import RandomFlip
from tensorflow.keras.layers import RandomZoom
from tensorflow.keras.layers import RandomRotation
from tensorflow.keras.layers import RandomBrightness
from tensorflow.keras.layers import RandomContrast
from tensorflow.keras.layers import RandomTranslation 

# Utilitaire d'importation des dataset d'images
from keras.utils import image_dataset_from_directory


# 2. Loading and Data Enrichment

In [ ]:
def classer_images_par_age(
    repertoire_images,
    age_min=None,
    age_max=None
):
    """
    Organise les images dans des sous-dossiers en fonction de l'âge extrait du nom de fichier.
    - Peut filtrer une plage d'âges : age_min à age_max
    - Réindexe les âges sélectionnés en commençant à 0
    """

    ages_valides = set()

    # Étape 1 — Parcourir tous les fichiers et collecter les âges valides
    fichiers_eligibles = []
    for racine, _, fichiers in os.walk(repertoire_images):
        for fichier in fichiers:
            if fichier.lower().endswith(".jpg"):
                try:
                    age = int(fichier.split("_")[0])
                    if ((age_min is None or age >= age_min) and
                        (age_max is None or age <= age_max)):
                        fichiers_eligibles.append((fichier, age, racine))
                        ages_valides.add(age)
                except Exception as e:
                    print(f"⚠️ Ignoré : {fichier} (erreur : {e})")

    # Étape 2 — Réindexer les âges valides
    ages_valides = sorted(ages_valides)
    mapping_ages = {age: idx for idx, age in enumerate(ages_valides)}
    print(f"🎯 Mapping des âges : {mapping_ages}")

    # Étape 3 — Déplacer les fichiers vers les bons dossiers (réindexés)
    for fichier, age, racine in fichiers_eligibles:
        nouvelle_classe = str(mapping_ages[age])
        dest_dir = os.path.join(repertoire_images, nouvelle_classe)
        os.makedirs(dest_dir, exist_ok=True)

        chemin_source = os.path.join(racine, fichier)
        chemin_destination = os.path.join(dest_dir, fichier)

        try:
            shutil.move(chemin_source, chemin_destination)
        except Exception as e:
            print(f"❌ Erreur déplacement {fichier} : {e}")

    print("✅ Organisation terminée.")

In [ ]:
data_dir = "C:\\Users\\remyc\\Downloads\\visages\\"  
min_age = 35
max_age = 42
classer_images_par_age(data_dir, min_age, max_age)

train_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,       # Fraction des données utilisée pour la validation
    subset="training",          # Charger les données partie entraînement
    seed=42,                    # Graine pour le découpage des données
    batch_size=64               # Taille des lots
)

val_ds = image_dataset_from_directory(
    data_dir,
    validation_split=0.2,       # Fraction des données utilisée pour la validation
    subset="validation",        # Charger les données partie validation
    seed=42,                    # même Graine pour récupérer les 20% restant 
    batch_size=64               # Taille des lots
)

In [ ]:
# Nombre de lot dans l'ensemble d'entraînement
print("Nombre de batch dans train_ds:", train_ds.cardinality().numpy())
# Nombre de lot dans l'ensemble de validation
print("Nombre de batch dans val_ds:", val_ds.cardinality().numpy())

In [ ]:
# Affichage aléatoire de 6 images
fig, axs = plt.subplots(2, 3, figsize=(14,9))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for images, labels in train_ds.take(1):
    for j, i in enumerate(np.random.choice(np.arange(0, len(labels)), size=6)):
        img = images[i].numpy().astype("uint8")
        axs[j].axis('off')
        # Affichage de l'image en niveaux de gris
        axs[j].imshow(img)
        # Titre avec le label
        axs[j].set_title(f'Label: {str(labels[i].numpy()+min_age)}')
plt.show()

# 3. Deep learning

>Bonnes pratiques (computer vision)
> - couches de transformation des images (redimensionnement, normalisation et augmentation)
> - couches convolutives de détection des features cachées (filtrage / bord / flou / ...)
> - couches de pooling (réduction de dimension)
> - couche de réduction des connections (pour éviter le surapprentissage)
> - couches denses d'apprentissage

## 3.1 Modèle Tensor Flow Keras (spécialisation image)

In [ ]:
class TimingCallback(Callback):
    def __init__(self, logs={}):
        self.logs=[]
    def on_epoch_begin(self, epoch, logs={}):
        self.starttime = timer()
    def on_epoch_end(self, epoch, logs={}):
        self.logs.append(timer()-self.starttime)

#### Creation & Compilation

In [ ]:
# callback optimisant le temps de traitement
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,                 # critère à observer sur 5 epochs
    min_delta=0.01,             # seuil de détection de critère fixé à 1% de variation
    verbose=1,                  # on log quand on arrête prématurément
    restore_best_weights=True,
    mode='min',                 
)
reduce_lr_on_plateau = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    patience=3,                 # critère à observer sur 3 epochs
    min_delta=0.01,             # seuil de détection de critère fixé à 1% de variation
    verbose=1,                  # on ne log que sur l'évènement de réduction
    factor=0.1,                 # facteur de réduction si le cycle est observé
    mode='min',
)
timing = TimingCallback()

In [ ]:
# Définitions des dimension d'entrée et de sortie optimisées
for images, labels in train_ds.take(1):
    shape_image = images.shape[1:]
    break # même si une seule image par sécurité...
print("Shape des images train :", shape_image)
num_classes = len(train_ds.class_names)
print("Nombre de classes train:", num_classes)

for images, labels in val_ds.take(1):
    shape_image = images.shape[1:]
    break # même si une seule image par sécurité...
print("Shape des images val :", shape_image)
num_classes = len(val_ds.class_names)
print("Nombre de classes val :", num_classes)

In [ ]:
### Instanciation du modèle par application successive des layers (fonctionnelle)

# Transformation des images : redimensionnement, normalisation et augmentation
inputs = Input(shape=shape_image, name="Input")

# x = Resizing(50,50)(inputs)     # Redimensionner les images à 50x50 pixels
# x = Rescaling(1./255)(x)        # Normalisation des pixels pour avoir des valeurs entre 0 et 1
# x = RandomFlip("horizontal")(x) # Retourner les images horizontalement de façon aléatoire
# x = RandomRotation(0.2)(x)      # Appliquer une rotation aléatoire entre -0.2 et +0.2
# x = RandomZoom(0.2)(x)          # Appliquer un zoom aléatoire entre 0.8 et 1.2
# x = RandomContrast(0.2)(x)      # Modifier le contraste de l'image de façon aléatoire entre -0.2 et +0.2
# x = RandomBrightness(0.1)(x)    # Appliquer une variation de la luminosité de l'image de -0.1 à +0.1
x = Conv2D(                     # 1ère Couche de convolution 3x3
    filters=32, 
    kernel_size=(3, 3), 
    padding='valid',
    strides=(1,1),
    activation="relu", 
    name="Convolution2D_1"
)(inputs)
x = MaxPooling2D(               # 1ère couche de pooling 2x2
    pool_size=(2, 2),
    name='Pooling2D_1'
)(x)
x = Dropout(rate=0.3)(x)        # Drop out anti overfitting (30% d'abandon de connexions)
x = Flatten()(x)                # Applatissement pour les couches denses
outputs = Dense(
    units=1, 
    activation='linear',
    name='Dense_output'
)(x)

nn_tfkeras_func = Model(inputs=inputs, outputs=outputs)

In [ ]:
nn_tfkeras_func.compile(
    loss='mse',                      # fonction de perte
    optimizer='adam',                # algorithme d'optimisation
    metrics=['mean_absolute_error'], # métrique d'évaluation
)

In [ ]:
training_history = nn_tfkeras_func.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=50, 
    callbacks = [
        reduce_lr_on_plateau,
        early_stopping,
        timing
    ]
)

In [ ]:
train_mae = training_history.history['mean_absolute_error']
val_mae = training_history.history['val_mean_absolute_error']
train_loss = training_history.history['loss']
val_loss = training_history.history['val_loss']
fig, axs = plt.subplots(1, 2, figsize=(12,6))
axs[0].plot(train_loss, label='Loss (training)')
axs[0].plot(val_loss, label='Loss (validation)')
axs[0].set_title('Loss evolution per epoch')
axs[0].set_xlabel('Epochs')
axs[0].set_ylabel('Loss')
axs[0].legend()
axs[1].plot(train_mae, label='MAE (training)')
axs[1].plot(val_mae, label='MAE (validation)')
axs[1].set_title('MAE evolution per epoch')
axs[1].set_xlabel('Epochs')
axs[1].set_ylabel('MAE')
axs[1].legend()
plt.show()

In [ ]:
# Prédictions du modèle : pour chaque échantillon, un vecteur de probabilités (1 par classe, grâce à softmax)
# Les classes sont ici codées de 0 à 9 (multi-classe)
y_test_prob = nn_tfkeras_func.predict(val_ds)
y_test_pred_class = np.rint(y_test_prob).astype(int).flatten()
y_test_pred_age = y_test_pred_class + min_age

y_test = []
for _, labels in val_ds.unbatch():
    y_test.append(labels.numpy())
y_test = np.array(y_test, dtype=int)
y_test_age = y_test + min_age

In [ ]:
# Étape 1 — Liste des erreurs grossières
error_indexes = []
for i in range(len(y_test_pred_age)):
    if (np.abs(y_test_pred_age[i] - y_test_age[i])>10):
        error_indexes += [i]
print(f"Nombre d'erreur grossières du modèle [{len(error_indexes)}]")

# Étape 2 — Récupération des images "flattened"
images_list = []
for batch_images, _ in val_ds.unbatch().batch(1).take(len(y_test_pred_age)):
    images_list.append(batch_images[0].numpy())  # batch_images est de forme (1, H, W, C)
images_array = np.array(images_list)  # (N, H, W, C)

# Affichage aléatoire des images sur lesquelles le modèle s'est trompé
(max_line, max_col) = (3, 3)
print(f"On en affiche aléatoirement {max_line}x{max_col}")
fig, axs = plt.subplots(max_line, max_col, figsize=(max_line*4,max_col*4))
# Aplatir la grille pour un accès linéaire
axs = axs.ravel()
for j, i in enumerate(np.random.choice(error_indexes, size=max_line*max_col, replace=False)):
    axs[j].axis('off')
    # Affichage de l'image en niveaux de gris
    axs[j].imshow(images_array[i].astype("float32") / 255.0)
    # Titre avec le label
    axs[j].set_title(
        f'True Label: {str(y_test_age[i])}\n'
        f'Prediction: {str(y_test_pred_age[i])}\n'
    )